In [32]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../tables/part_1")

consumer_path = DATA_DIR / "tbl_consumer.csv"
consumer_details_path = DATA_DIR / "consumer_user_details.parquet"
merchant_path = DATA_DIR / "tbl_merchants.parquet"

In [33]:
consumer = pd.read_csv(
    consumer_path,
    sep="|"
)

consumer_details = pd.read_parquet(
    consumer_details_path
)

merchants = pd.read_parquet(
    merchant_path
)

In [34]:
def clean_consumer(df):
    df = df.copy()

    # Postcodes are identifiers rather than numeric measurements.
    # Restore leading zeroes lost during CSV parsing.
    df["postcode"] = (
        df["postcode"]
        .astype(str)
        .str.zfill(4)
    )

    return df

In [35]:
def validate_merchant_fraud(df):
    # Required fields must be present
    assert df["merchant_abn"].notna().all(), \
        "Missing merchant ABN"

    assert df["order_datetime"].notna().all(), \
        "Missing order datetime"

    assert df["fraud_probability"].notna().all(), \
        "Missing fraud probability"

    # Fraud probability must be on the observed 0-100 scale
    assert df["fraud_probability"].between(0, 100).all(), \
        "Fraud probability outside 0-100 range"

    # Dates should have been converted during cleaning
    assert pd.api.types.is_datetime64_any_dtype(df["order_datetime"]), \
        "order_datetime is not datetime"

In [36]:
# validation functions 
VALID_STATES = {
    "ACT", "NSW", "NT", "QLD",
    "SA", "TAS", "VIC", "WA"
}

VALID_GENDERS = {
    "Male", "Female", "Undisclosed"
}


def validate_consumer(df):
    assert df["consumer_id"].notna().all(), \
        "Missing consumer IDs detected"

    assert df["consumer_id"].is_unique, \
        "Duplicate consumer IDs detected"

    assert df["state"].isin(VALID_STATES).all(), \
        "Invalid state detected"

    assert df["gender"].isin(VALID_GENDERS).all(), \
        "Invalid gender detected"

    assert df["postcode"].str.len().eq(4).all(), \
        "Invalid postcode length detected"

    assert df["name"].str.strip().ne("").all(), \
        "Blank consumer name detected"

    assert df["address"].str.strip().ne("").all(), \
        "Blank consumer address detected"

In [37]:
#consumer details 
def validate_consumer_details(df):
    assert df["consumer_id"].notna().all()
    assert df["user_id"].notna().all()

    assert df["consumer_id"].is_unique, \
        "Duplicate consumer IDs in mapping table"

    assert df["user_id"].is_unique, \
        "Duplicate user IDs in mapping table"
    
# merchant details
def validate_merchants(df):
    assert df.index.notna().all(), \
        "Missing merchant ABN"

    assert df.index.is_unique, \
        "Duplicate merchant ABN"

    assert (
        pd.Series(df.index.astype(str))
        .str.len()
        .eq(11)
        .all()
    ), "Invalid merchant ABN length"

    assert df["name"].notna().all(), \
        "Missing merchant names"

    assert df["name"].str.strip().ne("").all(), \
        "Blank merchant names"
    

# relationship validation 
def validate_consumer_relationship(consumer, consumer_details):

    consumer.merge(
        consumer_details,
        on="consumer_id",
        validate="one_to_one"
    )

    assert set(consumer["consumer_id"]) == \
           set(consumer_details["consumer_id"]), \
           "Consumer IDs do not match between tables"

In [38]:
# loading functions 
def load_consumer(path):
    return pd.read_csv(path, sep="|")


def load_consumer_details(path):
    return pd.read_parquet(path)


def load_merchants(path):
    return pd.read_parquet(path)


def load_consumer_fraud(path):
    return pd.read_csv(path)


def load_merchant_fraud(path):
    return pd.read_csv(path)

In [39]:
def clean_consumer_fraud(df):
    df = df.copy()

    # Remove exact duplicate fraud observations
    df = df.drop_duplicates()

    # Convert dates to datetime
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"],
        errors="raise"
    )

    return df

In [40]:
def clean_merchant_fraud(df):
    df = df.copy()

    # Convert dates to datetime
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"],
        errors="raise"
    )

    return df

In [41]:
def validate_consumer_fraud(df):
    assert df["user_id"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(0, 100).all()

    assert df.duplicated().sum() == 0

In [28]:
consumer_clean = clean_consumer(consumer)
consumer_clean["postcode"].str.len().value_counts()

postcode
4    499999
Name: count, dtype: int64

In [42]:
#final one 
def run_part1_pipeline():

    # --------------------
    # LOAD
    # --------------------

    consumer = pd.read_csv(
        DATA_DIR / "tbl_consumer.csv",
        sep="|"
    )

    consumer_details = pd.read_parquet(
        DATA_DIR / "consumer_user_details.parquet"
    )

    merchants = pd.read_parquet(
        DATA_DIR / "tbl_merchants.parquet"
    )

    consumer_fraud = pd.read_csv(
        DATA_DIR / "consumer_fraud_probability.csv"
    )

    merchant_fraud = pd.read_csv(
        DATA_DIR / "merchant_fraud_probability.csv"
    )


    # --------------------
    # CLEAN
    # --------------------

    consumer = clean_consumer(consumer)

    consumer_fraud = clean_consumer_fraud(
        consumer_fraud
    )

    merchant_fraud = clean_merchant_fraud(
        merchant_fraud
    )


    # --------------------
    # VALIDATE
    # --------------------

    validate_consumer(consumer)

    validate_consumer_details(
        consumer_details
    )

    validate_merchants(
        merchants
    )

    validate_consumer_fraud(
        consumer_fraud
    )

    validate_merchant_fraud(
        merchant_fraud
    )

    validate_consumer_relationship(
        consumer,
        consumer_details
    )


    # --------------------
    # RETURN CLEAN TABLES
    # --------------------

    return (
        consumer,
        consumer_details,
        merchants,
        consumer_fraud,
        merchant_fraud
    )

In [43]:
(
    consumer_clean,
    consumer_details_clean,
    merchants_clean,
    consumer_fraud_clean,
    merchant_fraud_clean
) = run_part1_pipeline()

In [44]:
consumer_enriched = consumer_clean.merge(
    consumer_details_clean,
    on="consumer_id",
    how="left",
    validate="one_to_one"
)

In [45]:
print(consumer_enriched.shape)
print(consumer_enriched["user_id"].isna().sum())

(499999, 7)
0


In [ ]:
"""
consumer                  -> cleaned + validated
consumer_user_details     -> validated
merchants                 -> validated
consumer_fraud            -> cleaned + validated
merchant_fraud            -> cleaned + validated
 
 and created a merged "consumer enriched" table with consumer and consumer_user_details
 

"""